# SciERC Relation Extraction with SciBERT

This notebook fine-tunes `allenai/scibert_scivocab_uncased` on SciERC relation extraction and exports artifacts for SciGraph.

In [ ]:
!pip -q install -U datasets transformers evaluate scikit-learn accelerate

import json
import os
from collections import Counter
from typing import Dict, Tuple

import numpy as np
from datasets import Dataset, load_dataset
from sklearn.metrics import classification_report, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments


In [ ]:
MODEL_NAME = "allenai/scibert_scivocab_uncased"
DATASET_CANDIDATES = [
    "hrithikpiyush/scierc",
    "nsusemiehl/SciERC",
]
OUT_DIR = "training/artifacts/scibert-scierc-relation"
RESULTS_PATH = "training/results.json"
os.makedirs("training/artifacts", exist_ok=True)

raw = None
selected_dataset = None
for ds_name in DATASET_CANDIDATES:
    try:
        raw = load_dataset(ds_name)
        selected_dataset = ds_name
        break
    except Exception as e:
        print(f"Could not load {ds_name}: {e}")

if raw is None:
    raise RuntimeError("None of the SciERC dataset mirrors could be loaded.")

if "validation" not in raw and "dev" in raw:
    raw = raw.copy()
    raw["validation"] = raw["dev"]

print("Loaded dataset:", selected_dataset)
print(raw)
print("train columns:", raw["train"].column_names)
print("sample row:", raw["train"][0])
raw


In [ ]:
def _get_tokens(row):
    tokens = row.get("tokens") or row.get("words")
    if tokens is None and "sentences" in row:
        s = row["sentences"]
        if isinstance(s, list) and s and isinstance(s[0], list):
            tokens = s[0]
        elif isinstance(s, list):
            tokens = s
    return tokens

def _get_entities(row):
    return row.get("ner") or row.get("entities") or []

def _get_relations(row):
    return row.get("relations") or row.get("relations_list") or []

def build_examples(split_rows):
    examples, labels = [], []
    for row in split_rows:
        tokens = _get_tokens(row)
        ents = _get_entities(row)
        rels = _get_relations(row)
        if not tokens:
            continue

        rel_lookup = {}
        for rel in rels:
            # expected: [head_start, head_end, tail_start, tail_end, relation_label]
            if isinstance(rel, (list, tuple)) and len(rel) >= 5:
                rel_lookup[(rel[0], rel[2])] = rel[4]

        for h in ents:
            for t in ents:
                if not isinstance(h, (list, tuple)) or not isinstance(t, (list, tuple)):
                    continue
                if len(h) < 2 or len(t) < 2 or h[0] == t[0]:
                    continue

                label = rel_lookup.get((h[0], t[0]), "NO_RELATION")
                h_text = " ".join(tokens[h[0]: h[1] + 1])
                t_text = " ".join(tokens[t[0]: t[1] + 1])
                sent_text = " ".join(tokens)
                examples.append(f"[HEAD] {h_text} [TAIL] {t_text} [SEP] {sent_text}")
                labels.append(label)

    return examples, labels

train_texts, train_labels = build_examples(raw["train"])
val_texts, val_labels = build_examples(raw["validation"])
test_texts, test_labels = build_examples(raw["test"])

print("train", len(train_texts), Counter(train_labels).most_common(8))
print("valid", len(val_texts), Counter(val_labels).most_common(8))
print("test", len(test_texts), Counter(test_labels).most_common(8))


In [ ]:
label_list = sorted(set(train_labels) | set(val_labels) | set(test_labels))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

train_ds = Dataset.from_dict({"text": train_texts, "label": [label2id[l] for l in train_labels]})
val_ds = Dataset.from_dict({"text": val_texts, "label": [label2id[l] for l in val_labels]})
test_ds = Dataset.from_dict({"text": test_texts, "label": [label2id[l] for l in test_labels]})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"additional_special_tokens": ["[HEAD]", "[TAIL]"]})

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

train_ds = train_ds.map(tok, batched=True)
val_ds = val_ds.map(tok, batched=True)
test_ds = test_ds.map(tok, batched=True)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    label2id=label2id,
    id2label=id2label,
)
model.resize_token_embeddings(len(tokenizer))
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    micro = f1_score(labels, preds, average="micro")
    macro = f1_score(labels, preds, average="macro")
    return {"micro_f1": micro, "macro_f1": macro}

args = TrainingArguments(
    output_dir="training/checkpoints/scierc-rel",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
test_metrics = trainer.evaluate(test_ds)
preds = trainer.predict(test_ds)
pred_ids = np.argmax(preds.predictions, axis=-1)

print("Test metrics:", test_metrics)
print(classification_report(test_ds["label"], pred_ids, target_names=[id2label[i] for i in range(len(id2label))], digits=4))

os.makedirs(OUT_DIR, exist_ok=True)
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

result_blob = {
    "model_name": MODEL_NAME,
    "dataset": DATASET_NAME,
    "labels": label_list,
    "test_metrics": test_metrics,
    "num_train_examples": len(train_ds),
    "num_validation_examples": len(val_ds),
    "num_test_examples": len(test_ds),
}

with open(RESULTS_PATH, "w") as f:
    json.dump(result_blob, f, indent=2)

print(f"Saved model to {OUT_DIR}")
print(f"Saved metrics to {RESULTS_PATH}")


In [ ]:
# Optional: zip artifacts for easy download from Colab
!cd training && zip -r scibert-scierc-artifacts.zip artifacts results.json
